In [1]:
# Cell 1: Import libraries
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

import joblib
import pickle
import time

print("✅ Libraries loaded successfully!")
print(f"TensorFlow version: {tf.__version__}")


✅ Libraries loaded successfully!
TensorFlow version: 2.18.0


In [5]:
## Cell 2: Load and Clean Dataset

# Cell 2: Load and clean dataset
print("="*60)
print("📊 LOADING DATASET")
print("="*60)

# Load dataset
df = pd.read_csv('sentiment_data.csv')
print(f"✅ Dataset loaded! Shape: {df.shape}")

# Check for missing values
print(f"\nMissing values:")
print(df.isnull().sum())

# Drop rows with missing reviews or sentiments
df = df.dropna(subset=['review', 'sentiment'])
print(f"\nAfter dropping missing values: {df.shape}")

# Clean dataset - remove duplicates
df = df.drop_duplicates(subset=['review'])
print(f"After removing duplicates: {df.shape}")

# Check class distribution
print("\n📊 Sentiment distribution:")
sentiment_counts = df['sentiment'].value_counts()
print(sentiment_counts)

# Remove classes with very few samples (less than 3)
min_samples = 3
valid_classes = sentiment_counts[sentiment_counts >= min_samples].index
df = df[df['sentiment'].isin(valid_classes)]
print(f"\nAfter removing rare classes: {df.shape}")

print("\n✅ Data cleaned successfully!")


📊 LOADING DATASET
✅ Dataset loaded! Shape: (76181, 2)

Missing values:
review       0
sentiment    0
dtype: int64

After dropping missing values: (76181, 2)
After removing duplicates: (76181, 2)

📊 Sentiment distribution:
sentiment
positive    38515
negative    36236
neutral      1430
Name: count, dtype: int64

After removing rare classes: (76181, 2)

✅ Data cleaned successfully!


In [6]:
## Cell 3: Preprocess Text

# Cell 3: Text preprocessing
print("="*60)
print("🔧 PREPROCESSING TEXT")
print("="*60)

def clean_text(text):
    """Clean and normalize text"""
    if pd.isna(text):
        return ""
    text = str(text)
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)  # Remove numbers and special chars
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Apply cleaning
print("🔄 Cleaning reviews...")
df['clean_review'] = df['review'].apply(clean_text)

# Remove empty reviews
df = df[df['clean_review'].str.len() > 0]

# Filter out very short reviews (less than 3 words)
df = df[df['clean_review'].str.split().str.len() >= 3]

print(f"✅ After cleaning: {df.shape}")

# Encode labels
label_encoder = LabelEncoder()
df['sentiment_encoded'] = label_encoder.fit_transform(df['sentiment'])

# Display sample
print("\n📝 Sample cleaned reviews:")
for i in range(min(3, len(df))):
    print(f"Original: {df.iloc[i]['review'][:50]}...")
    print(f"Cleaned: {df.iloc[i]['clean_review'][:50]}...")
    print(f"Sentiment: {df.iloc[i]['sentiment']}")
    print("-"*40)
    

🔧 PREPROCESSING TEXT
🔄 Cleaning reviews...
✅ After cleaning: (75571, 3)

📝 Sample cleaned reviews:
Original: 1 elevator not working....
Cleaned: elevator not working...
Sentiment: negative
----------------------------------------
Original: 1 euros to park per day on their own premises....
Cleaned: euros to park per day on their own premises...
Sentiment: neutral
----------------------------------------
Original: 1 hour queue to check-in....
Cleaned: hour queue to check in...
Sentiment: negative
----------------------------------------


In [7]:
## Cell 4: Train-Test Split

# Cell 4: Split dataset
print("="*60)
print("📊 SPLITTING DATASET")
print("="*60)

X = df['clean_review'].values
y = df['sentiment_encoded'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"✅ Training samples: {len(X_train)}")
print(f"✅ Testing samples: {len(X_test)}")

print(f"\n📊 Training distribution:")
print(pd.Series(y_train).value_counts().sort_index())
print(f"\n📊 Testing distribution:")
print(pd.Series(y_test).value_counts().sort_index())


📊 SPLITTING DATASET
✅ Training samples: 60456
✅ Testing samples: 15115

📊 Training distribution:
0    28848
1     1125
2    30483
Name: count, dtype: int64

📊 Testing distribution:
0    7212
1     282
2    7621
Name: count, dtype: int64


In [8]:
## Cell 5: Model 1 - Logistic Regression (Fastest)

# Cell 5: Model 1 - Logistic Regression with TF-IDF
print("="*60)
print("🚀 MODEL 1: Logistic Regression")
print("="*60)

start_time = time.time()

# Vectorize
print("🔄 Vectorizing text...")
tfidf = TfidfVectorizer(
    max_features=3000,
    ngram_range=(1, 2),
    stop_words='english',
    min_df=2  # Ignore rare words
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print(f"✅ Features: {X_train_tfidf.shape[1]}")

# Train
print("🔄 Training model...")
lr = LogisticRegression(
    C=2.0,
    solver='saga',
    max_iter=500,
    random_state=42,
    multi_class='multinomial',
    n_jobs=-1
)
lr.fit(X_train_tfidf, y_train)

# Evaluate
y_pred_lr = lr.predict(X_test_tfidf)
accuracy_lr = accuracy_score(y_test, y_pred_lr)
_, _, f1_lr, _ = precision_recall_fscore_support(y_test, y_pred_lr, average='weighted')

train_time = time.time() - start_time

print(f"\n✅ Training complete! Time: {train_time:.2f} seconds")
print(f"📊 Accuracy: {accuracy_lr:.4f}")
print(f"📊 F1-Score: {f1_lr:.4f}")

# Save
joblib.dump(lr, 'logistic_regression.pkl')
joblib.dump(tfidf, 'tfidf_vectorizer.pkl')
print("💾 Model saved!")


🚀 MODEL 1: Logistic Regression
🔄 Vectorizing text...
✅ Features: 3000
🔄 Training model...

✅ Training complete! Time: 54.54 seconds
📊 Accuracy: 0.8539
📊 F1-Score: 0.8475
💾 Model saved!


In [10]:
# Cell 6: Model 2 - Bidirectional LSTM (Fixed)
print("="*60)
print("🚀 MODEL 2: Bidirectional LSTM")
print("="*60)

start_time = time.time()

# Tokenize
print("🔄 Tokenizing text...")
tokenizer = Tokenizer(num_words=5000, oov_token='<OOV>')
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

max_len = 60
X_train_pad = pad_sequences(X_train_seq, maxlen=max_len, padding='post', truncating='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len, padding='post', truncating='post')

vocab_size = min(len(tokenizer.word_index) + 1, 5001)

# One-hot encode labels
y_train_cat = tf.keras.utils.to_categorical(y_train, num_classes=len(label_encoder.classes_))
y_test_cat = tf.keras.utils.to_categorical(y_test, num_classes=len(label_encoder.classes_))

print(f"✅ Vocabulary size: {vocab_size}")

# Build model (fixed)
print("🔄 Building LSTM model...")
model = Sequential([
    Embedding(vocab_size, 64, input_length=max_len),
    Bidirectional(LSTM(32, dropout=0.2, return_sequences=True)),
    Bidirectional(LSTM(16, dropout=0.2)),
    Dropout(0.3),
    Dense(16, activation='relu'),
    Dropout(0.3),
    Dense(len(label_encoder.classes_), activation='softmax')
])

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("✅ Model built!")

# Train
print("🔄 Training LSTM...")
callbacks = [
    EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6)
]

history = model.fit(
    X_train_pad, y_train_cat,
    epochs=15,
    batch_size=64,
    validation_split=0.2,
    callbacks=callbacks,
    verbose=1
)

# Evaluate
y_pred_lstm_probs = model.predict(X_test_pad, verbose=0)
y_pred_lstm = np.argmax(y_pred_lstm_probs, axis=1)
y_test_lstm_labels = np.argmax(y_test_cat, axis=1)

accuracy_lstm = accuracy_score(y_test_lstm_labels, y_pred_lstm)
_, _, f1_lstm, _ = precision_recall_fscore_support(y_test_lstm_labels, y_pred_lstm, average='weighted')

train_time = time.time() - start_time

print(f"\n✅ Training complete! Time: {train_time:.2f} seconds")
print(f"📊 Accuracy: {accuracy_lstm:.4f}")
print(f"📊 F1-Score: {f1_lstm:.4f}")

# Save
model.save('lstm_model.h5')
with open('lstm_tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)
print("💾 Model saved!")

🚀 MODEL 2: Bidirectional LSTM
🔄 Tokenizing text...
✅ Vocabulary size: 5001
🔄 Building LSTM model...
✅ Model built!
🔄 Training LSTM...
Epoch 1/15
756/756 ━━━━━━━━━━━━━━━━━━━━ 55s 61ms/step - accuracy: 0.7088 - loss: 0.6211 - val_accuracy: 0.8072 - val_loss: 0.4453 - learning_rate: 0.0010
Epoch 2/15
756/756 ━━━━━━━━━━━━━━━━━━━━ 49s 65ms/step - accuracy: 0.8215 - loss: 0.4385 - val_accuracy: 0.8233 - val_loss: 0.4184 - learning_rate: 0.0010
Epoch 3/15
756/756 ━━━━━━━━━━━━━━━━━━━━ 80s 63ms/step - accuracy: 0.8460 - loss: 0.3894 - val_accuracy: 0.8216 - val_loss: 0.4148 - learning_rate: 0.0010
Epoch 4/15
756/756 ━━━━━━━━━━━━━━━━━━━━ 50s 66ms/step - accuracy: 0.8602 - loss: 0.3508 - val_accuracy: 0.8193 - val_loss: 0.4237 - learning_rate: 0.0010
Epoch 5/15
756/756 ━━━━━━━━━━━━━━━━━━━━ 49s 65ms/step - accuracy: 0.8734 - loss: 0.3191 - val_accuracy: 0.8222 - val_loss: 0.4504 - learning_rate: 0.0010
Epoch 6/15
756/756 ━━━━━━━━━━━━━━━━━━━━ 50s 66ms/step - accuracy: 0.8963 - loss: 0.2709 - val_ac


✅ Training complete! Time: 356.61 seconds
📊 Accuracy: 0.8180
📊 F1-Score: 0.8099
💾 Model saved!


In [11]:
## Cell 7: Compare Models

# Cell 7: Model Comparison
print("="*60)
print("📊 MODEL COMPARISON")
print("="*60)

comparison = pd.DataFrame({
    'Model': ['Logistic Regression', 'LSTM'],
    'Accuracy': [accuracy_lr, accuracy_lstm],
    'F1-Score': [f1_lr, f1_lstm]
})

print("\n📊 Comparison Table:")
print(comparison.to_string(index=False))

# Find best
best_idx = np.argmax([f1_lr, f1_lstm])
best_model_name = ['Logistic Regression', 'LSTM'][best_idx]
best_f1 = [f1_lr, f1_lstm][best_idx]
best_acc = [accuracy_lr, accuracy_lstm][best_idx]

print(f"\n{'='*60}")
print(f"🏆 BEST MODEL: {best_model_name}")
print(f"   F1-Score: {best_f1:.4f}")
print(f"   Accuracy: {best_acc:.4f}")
print(f"{'='*60}")

# Save best model info
best_info = {
    'best_model': best_model_name,
    'f1_score': float(best_f1),
    'accuracy': float(best_acc)
}
with open('best_model_info.pkl', 'wb') as f:
    pickle.dump(best_info, f)
joblib.dump(label_encoder, 'label_encoder.pkl')
print("💾 Best model info saved!")

# Show classification report for best model
print("\n📊 Classification Report (Best Model):")
print("="*60)
if best_model_name == 'Logistic Regression':
    print(classification_report(y_test, y_pred_lr, target_names=label_encoder.classes_))
else:
    print(classification_report(y_test_lstm_labels, y_pred_lstm, target_names=label_encoder.classes_))


📊 MODEL COMPARISON

📊 Comparison Table:
              Model  Accuracy  F1-Score
Logistic Regression  0.853920  0.847484
               LSTM  0.817995  0.809880

🏆 BEST MODEL: Logistic Regression
   F1-Score: 0.8475
   Accuracy: 0.8539
💾 Best model info saved!

📊 Classification Report (Best Model):
              precision    recall  f1-score   support

    negative       0.85      0.86      0.85      7212
     neutral       0.39      0.05      0.08       282
    positive       0.86      0.88      0.87      7621

    accuracy                           0.85     15115
   macro avg       0.70      0.59      0.60     15115
weighted avg       0.85      0.85      0.85     15115



In [12]:
## Cell 8: Test Predictions

# Cell 8: Test Predictions
print("="*60)
print(f"🧪 TESTING BEST MODEL: {best_model_name}")
print("="*60)

# Load best model
label_encoder = joblib.load('label_encoder.pkl')

if best_model_name == 'Logistic Regression':
    lr = joblib.load('logistic_regression.pkl')
    tfidf = joblib.load('tfidf_vectorizer.pkl')
    
    def predict(text):
        clean = clean_text(text)
        vec = tfidf.transform([clean])
        pred = lr.predict(vec)[0]
        return label_encoder.inverse_transform([pred])[0]
else:
    model = tf.keras.models.load_model('lstm_model.h5')
    with open('lstm_tokenizer.pkl', 'rb') as f:
        tokenizer = pickle.load(f)
    max_len = 60
    
    def predict(text):
        clean = clean_text(text)
        seq = tokenizer.texts_to_sequences([clean])
        pad = pad_sequences(seq, maxlen=max_len, padding='post', truncating='post')
        probs = model.predict(pad, verbose=0)
        pred = np.argmax(probs, axis=1)[0]
        return label_encoder.inverse_transform([pred])[0]

# Test reviews
test_reviews = [
    "The room was clean and comfortable, staff were very friendly.",
    "The check-in was slow and the receptionist was rude.",
    "Breakfast was okay but nothing special.",
    "Location was perfect, walking distance to all attractions.",
    "The room was dirty and the air conditioning didn't work.",
    "Good value for money, would stay again.",
    "The hotel is in a noisy area, couldn't sleep at night.",
    "Excellent service and delicious food."
]

print("\n📝 Sample Predictions:")
print("-"*70)
for review in test_reviews:
    sentiment = predict(review)
    print(f"Review: {review[:50]}...")
    print(f"Predicted: {sentiment}")
    print("-"*70)


🧪 TESTING BEST MODEL: Logistic Regression

📝 Sample Predictions:
----------------------------------------------------------------------
Review: The room was clean and comfortable, staff were ver...
Predicted: positive
----------------------------------------------------------------------
Review: The check-in was slow and the receptionist was rud...
Predicted: negative
----------------------------------------------------------------------
Review: Breakfast was okay but nothing special....
Predicted: neutral
----------------------------------------------------------------------
Review: Location was perfect, walking distance to all attr...
Predicted: positive
----------------------------------------------------------------------
Review: The room was dirty and the air conditioning didn't...
Predicted: negative
----------------------------------------------------------------------
Review: Good value for money, would stay again....
Predicted: positive
----------------------------------------

In [ ]:

## Cell 9: Quick Single Prediction (Optional)

# Cell 9: Quick single prediction
def quick_predict(text):
    """Quick prediction function"""
    return predict(text)

# Example usage
print("Enter your review or type 'quit' to exit:")
while True:
    user_input = input("\nReview: ")
    if user_input.lower() == 'quit':
        break
    try:
        result = quick_predict(user_input)
        print(f"Sentiment: {result}")
    except Exception as e:
        print(f"Error: {e}")

Enter your review or type 'quit' to exit:



Review:  the ebnvironment hae a lot of noise and the food was saklty 


Sentiment: negative



Review:  bad environment


Sentiment: negative


...

Sentiment: positive



Review:  the food is delicious but the environment is very bad 


Sentiment: negative



Review:  unsafe 


Sentiment: negative
